In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

!pip uninstall -y torch torchaudio torchvision vllm

!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -q --no-index --find-links=/kaggle/input/vllm-wheels-cache vllm

!pip install -q sentence-transformers transformers accelerate

In [ ]:
!pip install torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu130

In [ ]:

!pip uninstall -y torch torchaudio torchvision

!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129

In [ ]:
!pip install vllm==0.11.0

In [ ]:
!pip install "transformers>=4.40.0,<5.0.0"

In [ ]:
import torch
import torchaudio

print(f"PyTorch: {torch.__version__}")
print(f"TorchAudio: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and "awq" in root.lower():
        print(root)
        break

/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1


In [4]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/foroughkoohi/graphrag-qa-data")
DST = Path("/kaggle/working")

for item in SRC.iterdir():
    dst = DST / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy(item, dst)

!ls -la /kaggle/working

total 36
drwxr-xr-x 7 root root 4096 Oct 14 10:59 .
drwxr-xr-x 5 root root 4096 Oct 14 10:03 ..
drwxr-xr-x 3 root root 4096 Oct 14 10:23 cache
drwxr-xr-x 3 root root 4096 Oct 14 10:24 data
drwxr-xr-x 3 root root 4096 Oct 14 10:59 runs
-rw-r--r-- 1 root root 8120 Oct 14 11:22 score.py
drwxr-xr-x 2 root root 4096 Oct 14 10:28 src
drwxr-xr-x 2 root root 4096 Oct 14 10:04 .virtual_documents


In [6]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # 🔑 این خط مشکل رو حل می‌کنه

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
OUT_DIR = Path("/kaggle/working/runs/hidden/closed_book")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
)
sampling = SamplingParams(temperature=0, max_tokens=30)

PROMPT_TEMPLATE = """Answer the question with a SHORT answer.
Rules:
- Use lowercase only
- No punctuation at the end
- For yes/no questions, answer exactly 'yes' or 'no'
- Do not explain
- Just give the answer

Question: {question}
Answer:"""

questions = [json.loads(l) for l in open(PACK / "questions_hidden.jsonl", encoding="utf-8")]
print(f"Number of questions: {len(questions)}")

prompts = [PROMPT_TEMPLATE.format(question=q["question"]) for q in questions]

t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"{time.time()-t0:.1f} seconds")

with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, out in zip(questions, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": [],
        }, ensure_ascii=False) + "\n")

print("Saved")

INFO 10-14 11:29:45 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'disable_log_stats': True, 'quantization': 'awq', 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-14 11:29:45 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 10-14 11:29:45 [model.py:1510] Using max model len 4096
INFO 10-14 11:29:45 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-14 11:29:55 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=1701) INFO 10-14 11:29:58 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1701) INFO 10-14 11:29:58 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, 

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.43s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.26it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.12it/s]
(EngineCore_DP0 pid=1701) 


(EngineCore_DP0 pid=1701) INFO 10-14 11:30:06 [default_loader.py:267] Loading weights took 1.79 seconds
(EngineCore_DP0 pid=1701) INFO 10-14 11:30:07 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 2.149946 seconds
(EngineCore_DP0 pid=1701) INFO 10-14 11:30:18 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/19b703ce77/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1701) INFO 10-14 11:30:18 [backends.py:559] Dynamo bytecode transform time: 10.48 s


(EngineCore_DP0 pid=1701) [rank0]:W1014 11:30:19.952000 1701 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


(EngineCore_DP0 pid=1701) INFO 10-14 11:30:23 [backends.py:197] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=1701) INFO 10-14 11:30:54 [backends.py:218] Compiling a graph for dynamic shape takes 34.88 s
(EngineCore_DP0 pid=1701) INFO 10-14 11:31:18 [monitor.py:34] torch.compile takes 45.36 s in total
(EngineCore_DP0 pid=1701) INFO 10-14 11:31:20 [gpu_worker.py:298] Available KV cache memory: 5.91 GiB
(EngineCore_DP0 pid=1701) INFO 10-14 11:31:21 [kv_cache_utils.py:1087] GPU KV cache size: 43,040 tokens
(EngineCore_DP0 pid=1701) INFO 10-14 11:31:21 [kv_cache_utils.py:1091] Maximum concurrency for 4,096 tokens per request: 10.51x
(EngineCore_DP0 pid=1701) WARNING 10-14 11:31:21 [gpu_model_runner.py:3663] CUDAGraphMode.FULL_AND_PIECEWISE is not supported with FlexAttentionMetadataBuilder backend (support: AttentionCGSupport.NEVER); setting cudagraph_mode=PIECEWISE because attention is compiled piecewise


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:28<00:00,  2.32it/s]


(EngineCore_DP0 pid=1701) INFO 10-14 11:31:52 [gpu_model_runner.py:3480] Graph capturing finished in 31 secs, took 0.80 GiB
(EngineCore_DP0 pid=1701) INFO 10-14 11:31:52 [core.py:210] init engine (profile, create kv cache, warmup model) took 105.18 seconds
INFO 10-14 11:31:54 [llm.py:306] Supported_tasks: ['generate']
Number of questions: 300


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

138.8 seconds
Saved


In [8]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import json
import time
from pathlib import Path
from vllm import LLM, SamplingParams

PACK = Path("/kaggle/working/data/course_pack_v1")
OUT_DIR = Path("/kaggle/working/runs/hidden/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = "/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1"

llm = LLM(
    model=MODEL_PATH,
    quantization="awq",
    dtype="half",
    max_model_len=4096,
    gpu_memory_utilization=0.85,   
    max_num_seqs=8,                 
    max_num_batched_tokens=4096,    
)
sampling = SamplingParams(temperature=0, max_tokens=30)
t0 = time.time()
outputs = llm.generate(prompts, sampling)
print(f"{time.time()-t0:.1f} seconds")

# Save
OUT_DIR = Path("/kaggle/working/runs/hidden/dense")
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for q, rids, out in zip(questions, retrieved_ids, outputs):
        answer = out.outputs[0].text.strip().split("\n")[0].strip().rstrip(".").lower()
        f.write(json.dumps({
            "qid": q["qid"],
            "answer": answer,
            "retrieved": rids,
        }, ensure_ascii=False) + "\n")

print("Saved")

INFO 10-14 11:38:03 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 4096, 'max_num_seqs': 8, 'disable_log_stats': True, 'quantization': 'awq', 'model': '/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1'}
INFO 10-14 11:38:03 [model.py:547] Resolved architecture: Qwen3ForCausalLM
INFO 10-14 11:38:03 [model.py:1510] Using max model len 4096
INFO 10-14 11:38:03 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 10-14 11:38:12 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:16 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:16 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/kaggle/input/models/alphadua/qwen3-8b-awq-community/transformers/1/1', speculative_config=None, tokenizer='/kaggle/input/models/alphadua/qwen3-8b-awq-community

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.45s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.25it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.11it/s]
(EngineCore_DP0 pid=2808) 


(EngineCore_DP0 pid=2808) INFO 10-14 11:38:23 [default_loader.py:267] Loading weights took 1.80 seconds
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:24 [gpu_model_runner.py:2653] Model loading took 5.7071 GiB and 2.162334 seconds
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:35 [backends.py:548] Using cache directory: /root/.cache/vllm/torch_compile_cache/19b703ce77/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:35 [backends.py:559] Dynamo bytecode transform time: 10.45 s
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:39 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 3.389 s
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:41 [monitor.py:34] torch.compile takes 10.45 s in total
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:46 [gpu_worker.py:298] Available KV cache memory: 6.13 GiB
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:47 [kv_cache_utils.py:1087] GPU KV cache size: 44,624 tokens
(EngineCore_DP0 pid=2808) INFO 10-14 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 5/5 [00:01<00:00,  3.04it/s]


(EngineCore_DP0 pid=2808) INFO 10-14 11:38:50 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 0.32 GiB
(EngineCore_DP0 pid=2808) INFO 10-14 11:38:50 [core.py:210] init engine (profile, create kv cache, warmup model) took 25.79 seconds
INFO 10-14 11:38:51 [llm.py:306] Supported_tasks: ['generate']


Adding requests:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore_DP0 pid=2808) CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 51 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


965.0 seconds
Saved


In [9]:
!zip -r runs.zip /kaggle/working/runs
from IPython.display import FileLink
FileLink("runs.zip")

  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/hidden/ (stored 0%)
  adding: kaggle/working/runs/hidden/closed_book/ (stored 0%)
  adding: kaggle/working/runs/hidden/closed_book/predictions.jsonl (deflated 83%)
  adding: kaggle/working/runs/hidden/dense/ (stored 0%)
  adding: kaggle/working/runs/hidden/dense/predictions.jsonl (deflated 73%)


/kaggle/working/runs.zip